<a href="https://colab.research.google.com/github/Dinazafreen/RAG--Based-Confidential-Knowledge-Assistant/blob/main/RAG_CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is still not enabled.")

CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

In [ ]:
def chatbot_response(message, history):
    response = f"You asked: {message}"
    return response

In [ ]:
demo = gr.ChatInterface(
    fn=chatbot_response,
    title="🔐 Confidential Knowledge Assistant",
    description="Ask questions based on the organization's authorized knowledge base.",
    textbox=gr.Textbox(
        placeholder="Ask your question here...",
        lines=2
    ),
    examples=[
        "What is the company leave policy?",
        "What are the working hours?",
        "How can I apply for leave?"
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e3e6b46829df2f398.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Add a Knowledge Base
#

In [ ]:
!pip install -q pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.6 MB/s eta 0:00:00


Uploading  knowledge-base document

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Company_Leave_Policy(1) (1).pdf to Company_Leave_Policy(1) (1).pdf


Checking  the uploaded file

In [ ]:
filename = list(uploaded.keys())[0]

print("Uploaded file:", filename)

Uploaded file: Company_Leave_Policy(1) (1).pdf


Extracting text from PDF

In [ ]:
from pypdf import PdfReader

reader = PdfReader(filename)

text = ""

for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text()

    if page_text:
        text += f"\n--- Page {page_number} ---\n"
        text += page_text

print(text[:5000])


--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.



In [ ]:
import os

# Store the uploaded PDF information
source_pdf_path = filename
source_pdf_name = os.path.basename(filename)

print("Source PDF:", source_pdf_name)
print("PDF path:", source_pdf_path)

Source PDF: Company_Leave_Policy(1) (1).pdf
PDF path: Company_Leave_Policy(1) (1).pdf


In [ ]:
print("📄 Source Document:")
print(source_pdf_name)
print("📃 Page: 1")

📄 Source Document:
Company_Leave_Policy(1) (1).pdf
📃 Page: 1


Installing LangChain text splitters

In [ ]:
!pip install -q langchain-text-splitters

Creating a basic chunker

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Number of chunks: 1

--- Chunk 1 ---
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.


Fixed-size chunking

In [ ]:
def fixed_size_chunking(text, chunk_size=500):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks


fixed_chunks = fixed_size_chunking(text, chunk_size=500)

print("Number of fixed-size chunks:", len(fixed_chunks))

for i, chunk in enumerate(fixed_chunks[:5]):
    print(f"\n--- Fixed Chunk {i + 1} ---")
    print(chunk)

Number of fixed-size chunks: 1

--- Fixed Chunk 1 ---

--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.



Recursive chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

recursive_chunks = recursive_splitter.split_text(text)

print("Number of recursive chunks:", len(recursive_chunks))

for i, chunk in enumerate(recursive_chunks[:5]):
    print(f"\n--- Recursive Chunk {i + 1} ---")
    print(chunk)

Number of recursive chunks: 1

--- Recursive Chunk 1 ---
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.


Compare the two


In [ ]:
print("========== CHUNKING COMPARISON ==========")

print("\nFixed-size chunking")
print("-------------------")
print("Number of chunks:", len(fixed_chunks))
print("Average characters:",
      sum(len(c) for c in fixed_chunks) / len(fixed_chunks))

print("\nRecursive chunking")
print("-------------------")
print("Number of chunks:", len(recursive_chunks))
print("Average characters:",
      sum(len(c) for c in recursive_chunks) / len(recursive_chunks))

========== CHUNKING COMPARISON ==========

Fixed-size chunking
-------------------
Number of chunks: 1
Average characters: 286.0

Recursive chunking
-------------------
Number of chunks: 1
Average characters: 284.0


Inspecting chunk quality

In [ ]:
print("========== FIXED-SIZE EXAMPLE ==========")

for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"\nChunk {i + 1}:")
    print(chunk)

print("\n\n========== RECURSIVE EXAMPLE ==========")

for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"\nChunk {i + 1}:")
    print(chunk)

========== FIXED-SIZE EXAMPLE ==========

Chunk 1:

--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.



========== RECURSIVE EXAMPLE ==========

Chunk 1:
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.


Semantic Chunking

Install Sentence Transformers

In [ ]:
!pip install -q sentence-transformers

Load the embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


Create sentences

In [ ]:
import re

sentences = re.split(r'(?<=[.!?])\s+', text)

sentences = [
    sentence.strip()
    for sentence in sentences
    if sentence.strip()
]

print("Number of sentences:", len(sentences))

for i, sentence in enumerate(sentences[:10]):
    print(f"{i + 1}. {sentence}")

Number of sentences: 4
1. --- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
2. Employees can apply for leave through the employee portal.
3. Unused leave can be carried forward according to company policy.
4. Working Hours
The standard working hours are 9:00 AM to 6:00 PM.


Generate sentence embeddings

In [ ]:
sentence_embeddings = embedding_model.encode(
    sentences,
    normalize_embeddings=True
)

print("Embedding shape:", sentence_embeddings.shape)

Embedding shape: (4, 384)


Create the FAISS vector database

Install FAISS

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.9 MB/s eta 0:00:00


Create embeddings for our chunks

In [ ]:
import numpy as np

chunk_embeddings = embedding_model.encode(
    recursive_chunks,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Number of chunks:", len(recursive_chunks))
print("Embedding shape:", chunk_embeddings.shape)

Number of chunks: 1
Embedding shape: (1, 384)


Create FAISS index

In [ ]:
import faiss

dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)

faiss_index.add(chunk_embeddings.astype("float32"))

print("FAISS index created successfully.")
print("Number of vectors in FAISS:", faiss_index.ntotal)

FAISS index created successfully.
Number of vectors in FAISS: 1


In [ ]:
import numpy as np
import faiss
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import os


# ============================================================
# GLOBAL RAG COMPONENTS
# ============================================================

text = ""

recursive_chunks = []

chunk_embeddings = None

faiss_index = None

filename = "No file loaded"

current_document_type = "public"


# ============================================================
# PROCESS UPLOADED DOCUMENT
# ============================================================

def process_uploaded_document(filepath_string, document_type="public"):

    global text
    global recursive_chunks
    global chunk_embeddings
    global faiss_index
    global filename
    global current_document_type
    global embedding_model


    if filepath_string is None:

        return (
            "Upload status: No file selected. "
            "Please upload a PDF document."
        )


    if not isinstance(filepath_string, str):

        return (
            "Upload status: Invalid file path."
        )


    if not os.path.exists(filepath_string):

        return (
            "Upload status: File does not exist."
        )


    try:

        # Store document information
        filename = os.path.basename(filepath_string)

        current_document_type = document_type


        # ====================================================
        # EXTRACT TEXT FROM PDF
        # ====================================================

        reader = PdfReader(filepath_string)

        current_text = ""


        for page_number, page in enumerate(
            reader.pages,
            start=1
        ):

            page_text = page.extract_text()


            if page_text:

                current_text += (
                    f"\n--- Page {page_number} ---\n"
                )

                current_text += page_text


        if not current_text.strip():

            return (
                "Upload status: No readable text found "
                "in the PDF."
            )


        text = current_text


        # ====================================================
        # CHUNK TEXT
        # ====================================================

        recursive_splitter = RecursiveCharacterTextSplitter(

            chunk_size=500,

            chunk_overlap=100,

            separators=[
                "\n\n",
                "\n",
                ". ",
                " ",
                ""
            ]
        )


        recursive_chunks = (
            recursive_splitter.split_text(text)
        )


        # ====================================================
        # LOAD EMBEDDING MODEL
        # ====================================================

        if "embedding_model" not in globals():

            embedding_model = SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2"
            )


        # ====================================================
        # CREATE EMBEDDINGS
        # ====================================================

        chunk_embeddings = embedding_model.encode(

            recursive_chunks,

            normalize_embeddings=True,

            convert_to_numpy=True
        )


        # ====================================================
        # CREATE FAISS INDEX
        # ====================================================

        dimension = chunk_embeddings.shape[1]


        faiss_index = faiss.IndexFlatIP(
            dimension
        )


        faiss_index.add(
            chunk_embeddings.astype("float32")
        )


        return (

            f"Upload status: Knowledge base updated successfully. "
            f"Document: {filename}. "
            f"Access level: {current_document_type}. "
            f"Total chunks: {len(recursive_chunks)}."

        )


    except Exception as e:

        return (
            f"Upload status: Error processing document: {str(e)}"
        )

In [ ]:
import os

filename_key = list(uploaded.keys())[0]
file_content_bytes = uploaded[filename_key]

# Save the bytes content to a local file
local_filepath = filename_key # Use the original filename
with open(local_filepath, "wb") as f:
    f.write(file_content_bytes)

# Call process_uploaded_document with the local filepath
status_message = process_uploaded_document(local_filepath)
print(status_message)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Upload status: Knowledge base updated with 1 chunks from 'Company_Leave_Policy(1) (1).pdf'. Access level: public


Search the knowledge base

Create a retrieval function

In [ ]:
def retrieve_documents(query, top_k=3):
    """
    Convert the user's query into an embedding
    and retrieve the most similar document chunks.
    """

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    scores, indices = faiss_index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for score, index in zip(scores[0], indices[0]):

        if index == -1:
            continue

        results.append({
            "chunk": recursive_chunks[index],
            "score": float(score),
            "index": int(index)
        })

    return results

Test retrieval

In [ ]:
query = "What is the leave policy?"

results = retrieve_documents(query, top_k=3)

for i, result in enumerate(results):

    print(f"\n===== Result {i + 1} ====")
    print("Score:", result["score"])
    print("Chunk index:", result["index"])
    print("Content:")
    print(result["chunk"])


===== Result 1 ====
Score: 0.5797512531280518
Chunk index: 0
Content:
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.


Test multiple questions

In [ ]:
test_questions = [
    "What is the leave policy?",
    "What are the working hours?",
    "How can employees apply for leave?"
]

for question in test_questions:

    print("\n" + "=" * 60)
    print("QUESTION:", question)
    print("=" * 60)

    results = retrieve_documents(question, top_k=2)

    for i, result in enumerate(results):

        print(f"\nResult {i + 1}")
        print("Score:", round(result["score"], 4))
        print("Chunk:")
        print(result["chunk"])


QUESTION: What is the leave policy?

Result 1
Score: 0.5798
Chunk:
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.

QUESTION: What are the working hours?

Result 1
Score: 0.4663
Chunk:
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.

QUESTION: How can employees apply for leave?

Result 1
Score: 0.528
Chunk:
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company poli

Install Transformers

In [ ]:
!pip install -q transformers sentencepiece

Load the LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("LLM loaded successfully:", model_name)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
print("Model device:", next(llm.parameters()).device)

Model device: cuda:0


In [ ]:
import time

print("Testing RAG with GPU...")

try:
    start = time.time()

    answer, results = rag_answer(
        "What are the working hours?",
        top_k=3
    )

    end = time.time()

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Sources:")
    print(results)

    print("\nTime taken:", round(end - start, 2), "seconds")

except Exception as e:
    print("\nRAG TEST FAILED")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

Testing RAG with GPU...

RAG TEST FAILED
Error type: NameError
Error details: name 'rag_answer' is not defined


Create a simple generation function

In [ ]:
def generate_answer(question, context):
    """
    Generate an answer using only the retrieved knowledge-base context.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "You are a confidential company knowledge assistant. "
                "Answer the user's question using ONLY the provided context. "
                "Do not use outside knowledge. "
                "Do not invent information. "
                "If the answer is not present in the context, say: "
                "\"I don't have enough information in the knowledge base to answer that.\""
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{question}

Give a clear and complete answer.
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [ ]:
test_context = """
Company Leave Policy

Employees are entitled to 18 days of annual leave per year.

Employees can apply for leave through the employee portal.

Unused leave can be carried forward according to company policy.

Working Hours

The standard working hours are 9:00 AM to 6:00 PM.
"""

questions = [
    "What is the leave policy?",
    "What are the working hours?",
    "How can employees apply for leave?"
]

for question in questions:

    answer = generate_answer(
        question,
        test_context
    )

    print("\nQuestion:", question)
    print("Answer:", answer)


Question: What is the leave policy?
Answer: The leave policy for employees at this company includes the following key points:

- Employees are entitled to 18 days of annual leave per year.
- Employees can apply for leave through the employee portal.
- Unused leave can be carried forward according to company policy.
- The standard working hours are from 9:00 AM to 6:00 PM.

Question: What are the working hours?
Answer: The standard working hours are from 9:00 AM to 6:00 PM.

Question: How can employees apply for leave?
Answer: Employees can apply for leave through the employee portal.


In [ ]:
questions = [
    "What is the leave policy?",
    "What are the working hours?",
    "How can employees apply for leave?"
]

for question in questions:

    print("\n" + "=" * 60)
    print("QUESTION:", question)

    try:
        result = rag_answer(
            question,
            top_k=3
        )

        # Check the returned result
        if isinstance(result, tuple):
            answer = result[0]
            results = result[1]
        else:
            answer = result
            results = []

        print("ANSWER:", answer)

        print("\nRETRIEVED SOURCES:")

        if results:
            for i, item in enumerate(results):
                print(f"\nSource {i + 1}")

                if isinstance(item, dict):
                    print("Score:", round(item.get("score", 0), 4))
                    print(item.get("chunk", "No chunk available"))
                else:
                    print(item)
        else:
            print("No retrieved sources returned.")

    except Exception as e:
        print("\nERROR OCCURRED")
        print("Error type:", type(e).__name__)
        print("Error details:", str(e))


QUESTION: What is the leave policy?

ERROR OCCURRED
Error type: NameError
Error details: name 'rag_answer' is not defined

QUESTION: What are the working hours?

ERROR OCCURRED
Error type: NameError
Error details: name 'rag_answer' is not defined

QUESTION: How can employees apply for leave?

ERROR OCCURRED
Error type: NameError
Error details: name 'rag_answer' is not defined


Testing the LLM separately

In [ ]:
context = """
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
"""

question = "How many days of annual leave are employees entitled to?"

answer = generate_answer(question, context)

print("Answer:")
print(answer)

Answer:
Employees are entitled to 18 days of annual leave per year.


In [ ]:
test_context = """
Company Leave Policy

Employees are entitled to 18 days of annual leave per year.

Employees can apply for leave through the employee portal.

Unused leave can be carried forward according to company policy.

Working Hours

The standard working hours are 9:00 AM to 6:00 PM.
"""

questions = [
    "What is the leave policy?",
    "What are the working hours?",
    "How can employees apply for leave?"
]

for question in questions:

    answer = generate_answer(
        question,
        test_context
    )

    print("\nQuestion:", question)
    print("Answer:", answer)


Question: What is the leave policy?
Answer: The leave policy for employees at this company includes the following key points:

- Employees are entitled to 18 days of annual leave per year.
- Employees can apply for leave through the employee portal.
- Unused leave can be carried forward according to company policy.
- The standard working hours are from 9:00 AM to 6:00 PM.

Question: What are the working hours?
Answer: The standard working hours are from 9:00 AM to 6:00 PM.

Question: How can employees apply for leave?
Answer: Employees can apply for leave through the employee portal.


In [ ]:
!pip install -q transformers accelerate

Creating  the RAG function

In [ ]:
def rag_answer(question, top_k=3):
    """
    Complete RAG pipeline:

    Question
       ↓
    FAISS retrieval
       ↓
    Relevant chunks
       ↓
    LLM
       ↓
    Answer
    """

    # Step 1: Retrieve relevant chunks
    results = retrieve_documents(
        question,
        top_k=top_k
    )

    # Step 2: Combine retrieved chunks
    context = "\n\n".join(
        result["chunk"]
        for result in results
    )

    # Step 3: Generate answer
    answer = generate_answer(
        question,
        context
    )

    return answer, results

Testing the first RAG answer

In [ ]:
question = "What is the leave policy?"

answer, results = rag_answer(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

print("\nSOURCES:")
for i, result in enumerate(results):
    print(f"\nSource {i + 1}")
    print("Score:", round(result["score"], 4))
    print(result["chunk"])

QUESTION:
What is the leave policy?

ANSWER:
The leave policy states that employees are entitled to 18 days of annual leave per year. Employees can apply for leave through the employee portal and unused leave can be carried forward according to company policy. The standard working hours are from 9:00 AM to 6:00 PM.

SOURCES:

Source 1
Score: 0.5798
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.


Creating the RAG chatbot function

Creating  the new chatbot interface

In [ ]:
def rag_chatbot(message, history):
    """
    Connect the Gradio chatbot to the RAG pipeline.
    """

    if not message or not message.strip():
        return "Please enter a question."

    answer, results = rag_answer(
        message,
        top_k=3
    )

    return answer

In [ ]:
import gradio as gr


# ============================================================
# UPLOAD DOCUMENT
# ============================================================

def upload_document(file, document_type):

    if file is None:
        return (
            "Please select a PDF document first.",
            None
        )

    try:
        # Process the uploaded document
        status = process_uploaded_document(
            file,
            document_type
        )

        # Return status and source file
        return status, file

    except Exception as e:
        return (
            f"Error while updating knowledge base: {str(e)}",
            None
        )


# ============================================================
# CHAT RESPONSE
# ============================================================

def respond(message, history, role, uploaded_file):

    if not message or not message.strip():
        return history, "", None

    # Check if a document is loaded
    if faiss_index is None or len(recursive_chunks) == 0:

        history = history + [
            {
                "role": "user",
                "content": message
            },
            {
                "role": "assistant",
                "content": "Please upload a document to the knowledge base first."
            }
        ]

        return history, "", None


    # ========================================================
    # SECURITY / PRIVACY CHECK
    # ========================================================

    if is_restricted_question(message):

        answer = (
            "Sorry, I cannot provide passwords, secrets, "
            "personal information, confidential information, "
            "or other restricted data."
        )

        history = history + [
            {
                "role": "user",
                "content": message
            },
            {
                "role": "assistant",
                "content": answer
            }
        ]

        return history, "", None


    # ========================================================
    # RBAC ACCESS CHECK
    # ========================================================

    if not check_access(role, current_document_type):

        answer = (
            "Access denied. You are not authorized "
            "to access this information."
        )

        history = history + [
            {
                "role": "user",
                "content": message
            },
            {
                "role": "assistant",
                "content": answer
            }
        ]

        return history, "", None


    # ========================================================
    # RAG ANSWER
    # ========================================================

    answer, results = rag_answer(
        message,
        top_k=3
    )


    # ========================================================
    # ADD SOURCE REFERENCE
    # ========================================================

    if results and len(results) > 0:

        source_text = (
            f"\n\n📄 Source: {filename}"
        )

    else:

        source_text = ""


    final_answer = answer + source_text


    # ========================================================
    # UPDATE CONVERSATION HISTORY
    # ========================================================

    history = history + [
        {
            "role": "user",
            "content": message
        },
        {
            "role": "assistant",
            "content": final_answer
        }
    ]


    # Show the source PDF
    return history, "", uploaded_file


# ============================================================
# GRADIO INTERFACE
# ============================================================

with gr.Blocks() as demo:

    gr.Markdown("# 🔐 Confidential Knowledge Assistant")

    gr.Markdown(
        "Ask questions based on the organization's knowledge base."
    )


    # ========================================================
    # ROLE SELECTION
    # ========================================================

    role = gr.Dropdown(
        choices=[
            "Admin",
            "HR",
            "Employee"
        ],
        value="Employee",
        label="Select Your Role"
    )


    # ========================================================
    # CHATBOT
    # ========================================================

    chatbot = gr.Chatbot(
        label="Conversation",
        height=400
    )


    # ========================================================
    # KNOWLEDGE BASE
    # ========================================================

    gr.Markdown("### Knowledge Base")

    uploaded_file = gr.File(
        label="Upload Document",
        type="filepath",
        file_types=[".pdf"]
    )


    # ========================================================
    # DOCUMENT ACCESS LEVEL - RADIO BUTTONS
    # ========================================================

    document_type = gr.Radio(
        choices=[
            "public",
            "employee",
            "hr",
            "confidential"
        ],
        value="public",
        label="Document Access Level"
    )


    upload_button = gr.Button(
        "Update Knowledge Base"
    )


    upload_status = gr.Textbox(
        label="Upload Status",
        interactive=False
    )


    # ========================================================
    # SOURCE DOCUMENT
    # ========================================================

    gr.Markdown("### 📄 Source Document")

    source_file = gr.File(
        label="Source used for the answer",
        interactive=False
    )


    # ========================================================
    # ASK QUESTION
    # ========================================================

    gr.Markdown("### Ask your question")

    with gr.Row():

        message = gr.Textbox(
            placeholder="Type your question here...",
            show_label=False,
            lines=2,
            scale=5
        )

        send_button = gr.Button(
            "Send",
            variant="primary",
            scale=1
        )


    clear_button = gr.Button("Clear Chat")


    # ========================================================
    # UPLOAD BUTTON EVENT
    # ========================================================

    upload_button.click(
        fn=upload_document,
        inputs=[
            uploaded_file,
            document_type
        ],
        outputs=[
            upload_status,
            source_file
        ]
    )


    # ========================================================
    # SEND BUTTON EVENT
    # ========================================================

    send_button.click(
        fn=respond,
        inputs=[
            message,
            chatbot,
            role,
            uploaded_file
        ],
        outputs=[
            chatbot,
            message,
            source_file
        ]
    )


    # ========================================================
    # ENTER KEY EVENT
    # ========================================================

    message.submit(
        fn=respond,
        inputs=[
            message,
            chatbot,
            role,
            uploaded_file
        ],
        outputs=[
            chatbot,
            message,
            source_file
        ]
    )


    # ========================================================
    # CLEAR CHAT EVENT
    # ========================================================

    clear_button.click(
        fn=lambda: ([], "", None),
        inputs=None,
        outputs=[
            chatbot,
            message,
            source_file
        ]
    )


# ============================================================
# LAUNCH APPLICATION
# ============================================================

demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://82847dd30be6b0e114.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import traceback # Import traceback to get detailed error info

def respond(message, history, role):
    if not message or not message.strip():
        return history, ""

    try:
        # Check if knowledge base is loaded
        if faiss_index is None or len(recursive_chunks) == 0:
            return (
                history + [
                    [
                        message,
                        "Please upload a document to the knowledge base first."
                    ]
                ],
                ""
            )

        # Check access using RBAC
        answer = rag_answer_with_role(
            message,
            role,
            "public"
        )

        # Add conversation to history
        history = history + [
            [
                message,
                answer
            ]
        ]

        return history, ""

    except Exception as e:
        error_message = f"An internal error occurred: {e}"
        print("--- Gradio Respond Error ---")
        print(error_message)
        traceback.print_exc() # Print full traceback to Colab console
        return (
            history + [
                [
                    message,
                    "Sorry, an internal error occurred while processing your request. Please check the Colab output for details."
                ]
            ],
            ""
        )

In [ ]:
import time

print("Testing RAG backend...")

start = time.time()

answer = rag_answer(
    "What are the working hours?",
    top_k=3
)

end = time.time()

print("Answer:")
print(answer)

print("\nTime taken:", round(end - start, 2), "seconds")

Testing RAG backend...
Answer:
('The standard working hours are from 9:00 AM to 6:00 PM.', [{'chunk': '--- Page 1 ---\nCompany Leave Policy\nEmployees are entitled to 18 days of annual leave per year.\nEmployees can apply for leave through the employee portal.\nUnused leave can be carried forward according to company policy.\nWorking Hours\nThe standard working hours are 9:00 AM to 6:00 PM.', 'score': 0.466323584318161, 'index': 0}])

Time taken: 2.33 seconds


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU is NOT enabled.")
    print("Please enable GPU from:")
    print("Runtime → Change runtime type → Hardware accelerator → T4 GPU")

CUDA available: True
GPU: Tesla T4


In [ ]:
def generate_answer(question, context):
    """
    Generate a complete answer using only the retrieved context.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful knowledge-base assistant. "
                "Answer the user's question using ONLY the information in the context below. "
                "Rules:\n1. Give a complete and clear sentence.\n2. Do not invent information.\n3. If the answer is not available in the context, say:\n   \"I don't have enough information in the knowledge base to answer that.\"\n4. Keep the answer concise."
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{question}
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(llm.device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    # The slice outputs[0][inputs["input_ids"].shape[1]:] should correctly remove the input prompt tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

The `rag_answer` function combines retrieval and generation. Let's define it now.

In [ ]:
def rag_answer(question, top_k=3):
    """
    Complete RAG pipeline:

    Question
       ↓
    FAISS retrieval
       ↓
    Relevant chunks
       ↓
    LLM
       ↓
    Answer
    """

    # Step 1: Retrieve relevant chunks
    results = retrieve_documents(
        question,
        top_k=top_k
    )

    # Step 2: Combine retrieved chunks
    context = "\n\n".join(
        result["chunk"]
        for result in results
    )

    # Step 3: Generate answer
    answer = generate_answer(
        question,
        context
    )

    return answer, results

In [ ]:
test_context = """
Company Leave Policy

Employees are entitled to 18 days of annual leave per year.

Employees can apply for leave through the employee portal.

Unused leave can be carried forward according to company policy.

Working Hours

The standard working hours are 9:00 AM to 6:00 PM.
"""

questions = [
    "What is the leave policy?",
    "What are the working hours?",
    "How can employees apply for leave?"
]

for question in questions:

    answer = generate_answer(
        question,
        test_context
    )

    print("\nContext:\n", test_context)
    print("\nQuestion:", question)
    print("Answer:", answer)


Context:
 
Company Leave Policy

Employees are entitled to 18 days of annual leave per year.

Employees can apply for leave through the employee portal.

Unused leave can be carried forward according to company policy.

Working Hours

The standard working hours are 9:00 AM to 6:00 PM.


Question: What is the leave policy?
Answer: Employees are entitled to 18 days of annual leave per year, which can be applied through the employee portal and carried forward according to company policy. The standard working hours are from 9:00 AM to 6:00 PM.

Context:
 
Company Leave Policy

Employees are entitled to 18 days of annual leave per year.

Employees can apply for leave through the employee portal.

Unused leave can be carried forward according to company policy.

Working Hours

The standard working hours are 9:00 AM to 6:00 PM.


Question: What are the working hours?
Answer: The standard working hours are 9:00 AM to 6:00 PM.

Context:
 
Company Leave Policy

Employees are entitled to 18 days

Role Based Access Control

In [ ]:
# ============================================
# RBAC - Role Based Access Control
# ============================================

ROLES = {
    "Admin": [
        "public",
        "employee",
        "hr",
        "confidential"
    ],

    "HR": [
        "public",
        "employee",
        "hr"
    ],

    "Employee": [
        "public",
        "employee"
    ]
}


def check_access(role, document_type):
    """
    Check whether a user role is allowed
    to access a particular type of document.
    """

    if role not in ROLES:
        return False

    return document_type in ROLES[role]


# Test RBAC
print("Admin → HR:", check_access("Admin", "hr"))
print("HR → HR:", check_access("HR", "hr"))
print("Employee → HR:", check_access("Employee", "hr"))
print("Employee → Public:", check_access("Employee", "public"))

Admin → HR: True
HR → HR: True
Employee → HR: False
Employee → Public: True


Connect RBAC to your RAG chatbot

In [ ]:
# ============================================
# RBAC + RAG Access Control
# ============================================

def rag_answer_with_role(question, role, document_type="public"):
    """
    Answer a question only if the user's role
    has permission to access the document.
    """

    # Check whether the role is allowed
    if not check_access(role, document_type):

        return (
            "Access denied. You are not authorized "
            "to access this information."
        )

    # If access is allowed, use the existing RAG pipeline
    answer, results = rag_answer(
        question,
        top_k=3
    )

    return answer


# ============================================
# Test RBAC + RAG
# ============================================

print("Employee asking about public information:")

answer = rag_answer_with_role(
    "What is the leave policy?",
    "Employee",
    "public"
)

print(answer)


print("\nEmployee asking about HR information:")

answer = rag_answer_with_role(
    "What is the HR policy?",
    "Employee",
    "hr"
)

print(answer)


print("\nAdmin asking about HR information:")

answer = rag_answer_with_role(
    "What is the HR policy?",
    "Admin",
    "hr"
)

print(answer)

Employee asking about public information:
Employees are entitled to 18 days of annual leave per year, which can be applied through the employee portal and carried forward according to company policy. Working hours are from 9:00 AM to 6:00 PM.

Employee asking about HR information:
Access denied. You are not authorized to access this information.

Admin asking about HR information:
The HR policy includes employees being entitled to 18 days of annual leave per year, the ability to apply for leave through an employee portal, and unused leave being carried forward according to company policy. Working hours are set from 9:00 AM to 6:00 PM.


In [ ]:
print("Testing RBAC + RAG...\n")

# Employee asking about public information
print("Employee → Public:")
print(
    rag_answer_with_role(
        "What are the working hours?",
        "Employee",
        "public"
    )
)

print("\n" + "=" * 60)

# Employee asking about HR information
print("Employee → HR:")
print(
    rag_answer_with_role(
        "What is the leave policy?",
        "Employee",
        "hr"
    )
)

print("\n" + "=" * 60)

# HR asking about HR information
print("HR → HR:")
print(
    rag_answer_with_role(
        "What is the leave policy?",
        "HR",
        "hr"
    )
)

print("\n" + "=" * 60)

# Admin asking about HR information
print("Admin → HR:")
print(
    rag_answer_with_role(
        "What is the leave policy?",
        "Admin",
        "hr"
    )
)

Testing RBAC + RAG...

Employee → Public:
The standard working hours are from 9:00 AM to 6:00 PM.

Employee → HR:
Access denied. You are not authorized to access this information.

HR → HR:
Employees are entitled to 18 days of annual leave per year, which can be applied through the employee portal and carried forward according to company policy. Working hours are from 9:00 AM to 6:00 PM.

Admin → HR:
Employees are entitled to 18 days of annual leave per year, which can be applied through the employee portal and carried forward according to company policy. Working hours are from 9:00 AM to 6:00 PM.


In [ ]:
answer, results = rag_answer(
    "What is the leave policy?",
    top_k=3
)

print("ANSWER:")
print(answer)

print("\n" + "=" * 60)
print("RETRIEVED SOURCES")
print("=" * 60)

for i, result in enumerate(results):

    print("\nSource", i + 1)

    print("Score:")
    print(result["score"])

    print("Chunk:")
    print(result["chunk"])

ANSWER:
Employees are entitled to 18 days of annual leave per year, which can be applied through the employee portal and carried forward according to company policy. Working hours are from 9:00 AM to 6:00 PM.

RETRIEVED SOURCES

Source 1
Score:
0.5797512531280518
Chunk:
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.


In [ ]:
def format_sources(results):
    """
    Format retrieved chunks as readable source references.
    """

    if not results:
        return ""

    sources_text = "\n\n📚 Sources:\n"

    for i, result in enumerate(results):

        chunk = result.get("chunk", "")
        score = result.get("score", 0)

        # Extract page number from the chunk
        page = "Unknown"

        if "--- Page " in chunk:
            try:
                page = chunk.split("--- Page ")[1].split(" ---")[0]
            except:
                page = "Unknown"

        sources_text += (
            f"\n📄 Source {i + 1}"
            f"\n   Page: {page}"
            f"\n   Similarity Score: {score:.4f}"
        )

    return sources_text

In [ ]:
answer, results = rag_answer(
    "What is the leave policy?",
    top_k=3
)

formatted_sources = format_sources(results)

print("ANSWER:")
print(answer)

print(formatted_sources)

ANSWER:
Employees are entitled to 18 days of annual leave per year, which can be applied through the employee portal and carried forward according to company policy. Working hours are from 9:00 AM to 6:00 PM.


📚 Sources:

📄 Source 1
   Page: 1
   Similarity Score: 0.5798


Conversation history

In [ ]:
def build_conversation_context(history):
    """
    Convert Gradio chat history into text
    that can be provided to the RAG system.
    """

    if not history:
        return ""

    context = ""

    for message in history:

        if isinstance(message, dict):

            role = message.get("role", "")
            content = message.get("content", "")

            if role == "user":
                context += f"User: {content}\n"

            elif role == "assistant":
                context += f"Assistant: {content}\n"

    return context

In [ ]:
test_history = [
    {
        "role": "user",
        "content": "What are the working hours?"
    },
    {
        "role": "assistant",
        "content": "The standard working hours are from 9:00 AM to 6:00 PM."
    }
]

conversation_context = build_conversation_context(test_history)

print("Conversation Context:")
print(conversation_context)

Conversation Context:
User: What are the working hours?
Assistant: The standard working hours are from 9:00 AM to 6:00 PM.



In [ ]:
def rag_answer_with_history(question, history, role, document_type="public"):
    """
    RAG answer using previous conversation history
    and role-based access control.
    """

    # Check RBAC first
    if not check_access(role, document_type):

        return (
            "Access denied. You are not authorized "
            "to access this information."
        )

    # Convert previous conversation into text
    conversation_context = build_conversation_context(history)

    # Retrieve relevant documents
    answer, results = rag_answer(
        question,
        top_k=3
    )

    # If there is no previous conversation,
    # return the normal RAG answer
    if not conversation_context.strip():
        return answer, results

    # Add conversation context to the answer prompt
    history_context = (
        "Previous conversation:\n"
        + conversation_context
        + "\n\nCurrent question:\n"
        + question
    )

    # Generate answer using the conversation-aware prompt
    final_answer, final_results = rag_answer(
        history_context,
        top_k=3
    )

    return final_answer, final_results

In [ ]:
test_history = [
    {
        "role": "user",
        "content": "What are the working hours?"
    },
    {
        "role": "assistant",
        "content": "The standard working hours are from 9:00 AM to 6:00 PM."
    }
]

answer, results = rag_answer_with_history(
    "Is this applicable to employees?",
    test_history,
    "Employee",
    "public"
)

print("FOLLOW-UP QUESTION:")
print("Is this applicable to employees?")

print("\nANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")

for i, result in enumerate(results):

    print(f"\nSource {i + 1}")
    print("Score:", round(result["score"], 4))
    print(result["chunk"])

FOLLOW-UP QUESTION:
Is this applicable to employees?

ANSWER:
Yes, this applies to all employees who work at the company.

RETRIEVED SOURCES:

Source 1
Score: 0.4175
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.


Security & Privacy Handling

In [ ]:
def security_check(question):
    """
    Detect requests for sensitive or restricted information.
    Returns True if the question is safe.
    """

    sensitive_keywords = [
        "password",
        "passwords",
        "secret",
        "secrets",
        "api key",
        "api keys",
        "access token",
        "access tokens",
        "private key",
        "private keys",
        "credit card",
        "bank account",
        "personal details",
        "personal information",
        "confidential information",
        "employee personal",
        "salary details",
        "ssn",
        "social security",
        "authentication token"
    ]

    question_lower = question.lower()

    for keyword in sensitive_keywords:

        if keyword in question_lower:
            return False

    return True

In [ ]:
test_questions = [
    "What are the working hours?",
    "What is the leave policy?",
    "What is the company password?",
    "Give me the employee personal details.",
    "What is the API key?"
]

for question in test_questions:

    allowed = security_check(question)

    print("\nQuestion:", question)

    if allowed:
        print("Result: ALLOWED")
    else:
        print("Result: BLOCKED")


Question: What are the working hours?
Result: ALLOWED

Question: What is the leave policy?
Result: ALLOWED

Question: What is the company password?
Result: BLOCKED

Question: Give me the employee personal details.
Result: BLOCKED

Question: What is the API key?
Result: BLOCKED


In [ ]:
import gradio as gr

print("PDF available for source reference:")
print(source_pdf_name)

PDF available for source reference:
Company_Leave_Policy(1) (1).pdf


In [ ]:
def create_source_reference(results):
    """
    Create a readable source reference from retrieved results.
    """

    if not results:
        return "No source available."

    references = []

    for i, result in enumerate(results):

        chunk = result.get("chunk", "")

        # Extract page number
        page = "Unknown"

        if "--- Page " in chunk:
            try:
                page = chunk.split("--- Page ")[1].split(" ---")[0]
            except:
                page = "Unknown"

        references.append(
            f"📄 {source_pdf_name} — Page {page}"
        )

    return "\n".join(references)


# Test the source reference

answer, results = rag_answer(
    "What are the working hours?",
    top_k=3
)

print("ANSWER:")
print(answer)

print("\nSOURCE REFERENCE:")
print(create_source_reference(results))

ANSWER:
The standard working hours are from 9:00 AM to 6:00 PM.

SOURCE REFERENCE:
📄 Company_Leave_Policy(1) (1).pdf — Page 1


In [ ]:
import os
import shutil

# Create a folder for source documents
source_folder = "/content/source_documents"
os.makedirs(source_folder, exist_ok=True)

# Copy the uploaded PDF into the source folder
source_pdf_copy = os.path.join(
    source_folder,
    source_pdf_name
)

shutil.copy2(
    source_pdf_path,
    source_pdf_copy
)

print("PDF copied successfully:")
print(source_pdf_copy)

PDF copied successfully:
/content/source_documents/Company_Leave_Policy(1) (1).pdf


In [ ]:
import urllib.parse

def create_clickable_source(results):
    """
    Create source references with PDF filename and page number.
    """

    if not results:
        return "No source available."

    references = []

    for i, result in enumerate(results):

        chunk = result.get("chunk", "")

        page = "Unknown"

        if "--- Page " in chunk:
            try:
                page = chunk.split("--- Page ")[1].split(" ---")[0]
            except:
                page = "Unknown"

        # Gradio-compatible file link
        file_url = f"/file={urllib.parse.quote(source_pdf_copy)}"

        references.append(
            f'📄 <a href="{file_url}" target="_blank">'
            f'{source_pdf_name} — Page {page}'
            f'</a>'
        )

    return "<br>".join(references)


# Test

answer, results = rag_answer(
    "What are the working hours?",
    top_k=3
)

print("ANSWER:")
print(answer)

print("\nCLICKABLE SOURCE:")
print(create_clickable_source(results))

ANSWER:
The standard working hours are from 9:00 AM to 6:00 PM.

CLICKABLE SOURCE:
📄 <a href="/file=/content/source_documents/Company_Leave_Policy%281%29%20%281%29.pdf" target="_blank">Company_Leave_Policy(1) (1).pdf — Page 1</a>


In [ ]:
# ============================================
# SECURITY AND PRIVACY CHECK
# ============================================

def is_restricted_question(question):

    if not question:
        return False

    question = question.lower()

    restricted_keywords = [
        "password",
        "passwords",
        "secret",
        "secrets",
        "confidential information",
        "personal information",
        "personal details",
        "bank details",
        "credit card",
        "private data",
        "private information",
        "login credentials",
        "credentials"
    ]

    for keyword in restricted_keywords:
        if keyword in question:
            return True

    return False

Create the semantic chunks

In [ ]:
# ============================================================
# SEMANTIC CHUNKING - COMPARISON EXPERIMENT
# ============================================================

import re

# Split document into sentences
sentences = re.split(r'(?<=[.!?])\s+', text)

sentences = [
    sentence.strip()
    for sentence in sentences
    if sentence.strip()
]

# Create semantic chunks by grouping complete sentences
semantic_chunks = []

current_chunk = ""

max_chunk_size = 500

for sentence in sentences:

    # Check whether the next sentence fits in the current chunk
    if len(current_chunk) + len(sentence) + 1 > max_chunk_size:

        if current_chunk.strip():
            semantic_chunks.append(current_chunk.strip())

        current_chunk = sentence

    else:

        current_chunk += " " + sentence


# Add the final chunk
if current_chunk.strip():
    semantic_chunks.append(current_chunk.strip())


# Display results
print("========== SEMANTIC CHUNKING ==========")

print("\nNumber of semantic chunks:", len(semantic_chunks))

for i, chunk in enumerate(semantic_chunks):
    print(f"\n--- Semantic Chunk {i + 1} ---")
    print(chunk)

========== SEMANTIC CHUNKING ==========

Number of semantic chunks: 1

--- Semantic Chunk 1 ---
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year. Employees can apply for leave through the employee portal. Unused leave can be carried forward according to company policy. Working Hours
The standard working hours are 9:00 AM to 6:00 PM.


Compare all three chunking approaches

In [ ]:
# ============================================================
# FINAL CHUNKING STRATEGY COMPARISON
# ============================================================

print("=" * 60)
print("FINAL CHUNKING STRATEGY COMPARISON")
print("=" * 60)

print("\n1. Fixed-Size Chunking")
print("Number of chunks:", len(fixed_chunks))
print(
    "Average chunk length:",
    round(
        sum(len(chunk) for chunk in fixed_chunks) / len(fixed_chunks),
        2
    )
)

print("\n2. Recursive Chunking")
print("Number of chunks:", len(recursive_chunks))
print(
    "Average chunk length:",
    round(
        sum(len(chunk) for chunk in recursive_chunks)
        / len(recursive_chunks),
        2
    )
)

print("\n3. Semantic Chunking")
print("Number of chunks:", len(semantic_chunks))
print(
    "Average chunk length:",
    round(
        sum(len(chunk) for chunk in semantic_chunks)
        / len(semantic_chunks),
        2
    )
)


print("\n" + "=" * 60)
print("OBSERVATION")
print("=" * 60)

print("""
For the current short document, all three chunking
strategies produced a single chunk.

Fixed-size chunking is simple and fast but can split
sentences or important context in larger documents.

Recursive chunking uses natural text boundaries and is
better at preserving context. It was selected for the
main RAG application.

Semantic chunking groups related sentences and can provide
better contextual grouping for larger and more complex
documents, but it may require additional embedding
computations.
""")

FINAL CHUNKING STRATEGY COMPARISON

1. Fixed-Size Chunking
Number of chunks: 1
Average chunk length: 286.0

2. Recursive Chunking
Number of chunks: 1
Average chunk length: 284.0

3. Semantic Chunking
Number of chunks: 1
Average chunk length: 284.0

OBSERVATION

For the current short document, all three chunking
strategies produced a single chunk.

Fixed-size chunking is simple and fast but can split
sentences or important context in larger documents.

Recursive chunking uses natural text boundaries and is
better at preserving context. It was selected for the
main RAG application.

Semantic chunking groups related sentences and can provide
better contextual grouping for larger and more complex
documents, but it may require additional embedding
computations.



Compare two embedding models

In [ ]:
# ============================================================
# EMBEDDING MODEL COMPARISON
# ============================================================

from sentence_transformers import SentenceTransformer
import time


# Test query
test_query = "What are the working hours?"


# ------------------------------------------------------------
# MODEL 1: Current model used in the RAG application
# ------------------------------------------------------------

print("Loading Model 1...")

model_1 = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

start_time = time.time()

query_embedding_1 = model_1.encode(
    test_query,
    normalize_embeddings=True,
    convert_to_numpy=True
)

time_1 = time.time() - start_time


# ------------------------------------------------------------
# MODEL 2: Alternative free embedding model
# ------------------------------------------------------------

print("Loading Model 2...")

model_2 = SentenceTransformer(
    "sentence-transformers/paraphrase-MiniLM-L3-v2"
)

start_time = time.time()

query_embedding_2 = model_2.encode(
    test_query,
    normalize_embeddings=True,
    convert_to_numpy=True
)

time_2 = time.time() - start_time


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EMBEDDING MODEL COMPARISON")
print("=" * 60)

print("\nModel 1")
print("Name: all-MiniLM-L6-v2")
print("Embedding dimension:", query_embedding_1.shape[0])
print("Encoding time:", round(time_1, 4), "seconds")


print("\nModel 2")
print("Name: paraphrase-MiniLM-L3-v2")
print("Embedding dimension:", query_embedding_2.shape[0])
print("Encoding time:", round(time_2, 4), "seconds")


print("\n" + "=" * 60)
print("OBSERVATION")
print("=" * 60)

print("""
all-MiniLM-L6-v2 provides high-quality general-purpose
sentence embeddings and was selected for the main RAG system.

paraphrase-MiniLM-L3-v2 is a smaller and lightweight model
that can be faster and requires fewer computational resources.

The all-MiniLM-L6-v2 model was retained for the final
application because it provides a good balance between
retrieval quality and performance.
""")

Loading Model 1...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading Model 2...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


EMBEDDING MODEL COMPARISON

Model 1
Name: all-MiniLM-L6-v2
Embedding dimension: 384
Encoding time: 0.0568 seconds

Model 2
Name: paraphrase-MiniLM-L3-v2
Embedding dimension: 384
Encoding time: 0.0096 seconds

OBSERVATION

all-MiniLM-L6-v2 provides high-quality general-purpose
sentence embeddings and was selected for the main RAG system.

paraphrase-MiniLM-L3-v2 is a smaller and lightweight model
that can be faster and requires fewer computational resources.

The all-MiniLM-L6-v2 model was retained for the final
application because it provides a good balance between
retrieval quality and performance.



retrieval comparison

In [ ]:
# ============================================================
# RETRIEVAL AND SCORING COMPARISON
# FAISS Similarity Search with Different Top-K Values
# ============================================================

import numpy as np


# Test question
test_query = "What are the working hours?"


# Use the embedding model from the main RAG system
query_embedding = embedding_model.encode(
    [test_query],
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")


# ============================================================
# TEST DIFFERENT TOP-K VALUES
# ============================================================

for top_k in [1, 2, 3]:

    print("\n" + "=" * 60)
    print(f"RETRIEVAL TEST - TOP K = {top_k}")
    print("=" * 60)

    # Make sure we don't request more results than available
    k = min(top_k, len(recursive_chunks))

    scores, indices = faiss_index.search(
        query_embedding,
        k
    )

    print("\nRetrieved chunks:")

    for rank, index in enumerate(indices[0], start=1):

        print(f"\nRank {rank}")
        print("Similarity Score:", round(float(scores[0][rank - 1]), 4))
        print("Chunk:")
        print(recursive_chunks[index])


RETRIEVAL TEST - TOP K = 1

Retrieved chunks:

Rank 1
Similarity Score: 0.4663
Chunk:
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.

RETRIEVAL TEST - TOP K = 2

Retrieved chunks:

Rank 1
Similarity Score: 0.4663
Chunk:
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave can be carried forward according to company policy.
Working Hours
The standard working hours are 9:00 AM to 6:00 PM.

RETRIEVAL TEST - TOP K = 3

Retrieved chunks:

Rank 1
Similarity Score: 0.4663
Chunk:
--- Page 1 ---
Company Leave Policy
Employees are entitled to 18 days of annual leave per year.
Employees can apply for leave through the employee portal.
Unused leave c

In [ ]:
# ============================================================
# RETRIEVAL / SCORING OBSERVATION
# ============================================================

print("=" * 60)
print("RETRIEVAL AND SCORING OBSERVATION")
print("=" * 60)

print("""
The RAG system uses SentenceTransformer embeddings together
with FAISS IndexFlatIP for similarity-based retrieval.

The embeddings are normalized before being stored and queried.
Therefore, the Inner Product similarity search is used to
identify the most relevant document chunks.

For the current test document, only one chunk is available,
so Top-K values of 1, 2, and 3 all returned the same available
chunk with a similarity score of approximately 0.4663.

For larger documents containing multiple chunks, Top-K retrieval
can return several relevant chunks. A lower Top-K value reduces
the amount of context sent to the language model, while a higher
Top-K value can provide more context but may also include less
relevant information.

FAISS IndexFlatIP was selected because it provides simple and
accurate similarity-based vector retrieval for the current
knowledge base.
""")

RETRIEVAL AND SCORING OBSERVATION

The RAG system uses SentenceTransformer embeddings together
with FAISS IndexFlatIP for similarity-based retrieval.

The embeddings are normalized before being stored and queried.
Therefore, the Inner Product similarity search is used to
identify the most relevant document chunks.

For the current test document, only one chunk is available,
so Top-K values of 1, 2, and 3 all returned the same available
chunk with a similarity score of approximately 0.4663.

For larger documents containing multiple chunks, Top-K retrieval
can return several relevant chunks. A lower Top-K value reduces
the amount of context sent to the language model, while a higher
Top-K value can provide more context but may also include less
relevant information.

FAISS IndexFlatIP was selected because it provides simple and
accurate similarity-based vector retrieval for the current
knowledge base.

